Jose Zavala

In [ ]:
import asyncio
import threading
import multiprocessing
import time
import math
import sys
import nest_asyncio


sys.set_int_max_str_digits(1_000_000)


nest_asyncio.apply()


def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True


def find_highest_prime(shared_prime, runtime_seconds=180):
    start_time = time.time()
    num = 0
    highest = 0
    while time.time() - start_time < runtime_seconds:
        if is_prime(num):
            highest = num
        num += 1
    shared_prime.value = highest
    print(f"Multiprocessing (7 cores): {highest:,}")


async def async_fibonacci(n):
    def fib(n):
        if n == 0:
            return (0, 1)
        a, b = fib(n >> 1)
        c = a * ((b << 1) - a)
        d = a * a + b * b
        return (c, d) if n & 1 == 0 else (d, c + d)

    print(f"[Async] Calculating Fibonacci({n})...")
    fib_n = fib(n)[0]
    print(f"[Async] Fibonacci({n}) has {len(str(fib_n)):,} digits")
    print(f"Async: {n:,}")


def threaded_factorial(n):
    print(f"[Thread] Calculating Factorial({n})...")
    result = math.factorial(n)
    print(f"[Thread] Factorial({n}) has {len(str(result)):,} digits")
    print(f"Threaded: {n:,}")


async def main_async():
    shared_prime = multiprocessing.Value('i', 0)
    runtime_seconds = 60  # 1 minute

  
    print("[Main] Starting multiprocessing prime search...")
    prime_proc = multiprocessing.Process(target=find_highest_prime, args=(shared_prime, runtime_seconds))
    prime_proc.start()
    prime_proc.join()

    highest_prime = shared_prime.value

    
    fib_input = 960_737
    fact_input = 10_747_921

    factorial_thread = threading.Thread(target=threaded_factorial, args=(fact_input,))
    factorial_thread.start()

    await async_fibonacci(fib_input)

    factorial_thread.join()
    print("\n[Main] All tasks completed.")


await main_async()

[Main] Starting multiprocessing prime search...
[Thread] Calculating Factorial(10747921)...
[Async] Calculating Fibonacci(960737)...


Exception in thread Thread-5 (threaded_factorial):
Traceback (most recent call last):
  File "c:\Users\josezavala\.anaconda3\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\josezavala\.anaconda3\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\josezavala\AppData\Local\Temp\ipykernel_16036\3458078461.py", line 55, in threaded_factorial
ValueError: Exceeds the limit (1000000 digits) for integer string conversion; use sys.set_int_max_str_digits() to increase the limit


[Async] Fibonacci(960737) has 200,782 digits
Async: 960,737

[Main] All tasks completed.


Used Chat GPT to help with code 